In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import json
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')


In [3]:
# (CHANGE THESE ACCORDING TO YOUR DRIVE)
STUDENTLIFE_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/data/raw/studentlife"
OUTPUT_PATH = "/content/drive/MyDrive/AI_Burnout_Predictor/results_realistic_studentlife"

os.makedirs(OUTPUT_PATH, exist_ok=True)

print("Paths configured.")


Paths configured.


In [4]:
# Loading StudentLife data
def load_studentlife_json(folder_path):
    all_data = []
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            student_id = file_name.replace(".json", "")
            with open(os.path.join(folder_path, file_name), "r") as f:
                records = json.load(f)
                for r in records:
                    r["student_id"] = student_id
                    all_data.append(r)
    return pd.DataFrame(all_data)

print("Loading StudentLife...")

stress_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Stress"))
activity_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Activity"))
sleep_raw = load_studentlife_json(os.path.join(STUDENTLIFE_PATH, "Sleep"))

print(f"Stress: {stress_raw.shape}")
print(f"Activity: {activity_raw.shape}")
print(f"Sleep: {sleep_raw.shape}")


Loading StudentLife...
Stress: (2408, 5)
Activity: (833, 9)
Sleep: (1644, 7)


In [5]:
stress_raw.head()

,null,resp_time,student_id,level,location
0,3,1364121467,Stress_u20,NaN,NaN
1,"43.70413179,-72.28882107",1364121469,Stress_u20,NaN,NaN
2,1,1364121470,Stress_u20,NaN,NaN
3,"43.70413179,-72.28882107",1364121793,Stress_u20,NaN,NaN
4,2,1364121465,Stress_u20,NaN,NaN


In [6]:
activity_raw.head()

,Social2,null,resp_time,student_id,other_relaxing,other_working,relaxing,working,location
0,2,4,1364884639,Activity_u33,NaN,NaN,NaN,NaN,NaN
1,3,2,1364590835,Activity_u33,NaN,NaN,NaN,NaN,NaN
2,2,1,1364504673,Activity_u33,NaN,NaN,NaN,NaN,NaN
3,2,2,1364677565,Activity_u33,NaN,NaN,NaN,NaN,NaN
4,2,2,1364765272,Activity_u33,NaN,NaN,NaN,NaN,NaN


In [7]:
sleep_raw.head()

,hour,location,rate,resp_time,social,student_id,null
0,9,"43.70357146,-72.29017646",1,1364761981,1,Sleep_u22,NaN
1,NaN,NaN,NaN,1364122237,NaN,Sleep_u22,6
2,NaN,NaN,NaN,1364122241,NaN,Sleep_u22,"43.70629505,-72.28825598"
3,NaN,NaN,NaN,1364122243,NaN,Sleep_u22,9
4,NaN,NaN,NaN,1364122235,NaN,Sleep_u22,6


In [8]:
# Cleaning the STRESS DATASET
# =========================
print("\n DATASET: STRESS")
print("Shows self-reported student stress levels over time")

print("\nCleaning StudentLife Stress...")

stress_clean = stress_raw.copy()

# Dropping the 'null' column
if 'null' in stress_clean.columns:
    stress_clean = stress_clean.drop(columns=['null'])

print("\nInitial Stress Dataset:")
print(stress_clean)

# Converting  timestamp
stress_clean['timestamp'] = pd.to_datetime(stress_clean['resp_time'], unit='s')
print("\nAfter converting resp_time to timestamp:")
print(stress_clean)

# Cleaning  student_id
stress_clean['student_id'] = stress_clean['student_id'].str.replace('Stress_', '')
print("\nAfter cleaning student_id:")
print(stress_clean)



 DATASET: STRESS
Shows self-reported student stress levels over time

Cleaning StudentLife Stress...

Initial Stress Dataset:
       resp_time  student_id level                  location
0     1364121467  Stress_u20   NaN                       NaN
1     1364121469  Stress_u20   NaN                       NaN
2     1364121470  Stress_u20   NaN                       NaN
3     1364121793  Stress_u20   NaN                       NaN
4     1364121465  Stress_u20   NaN                       NaN
...          ...         ...   ...                       ...
2403  1366085234  Stress_u41     1  43.70428777,-72.28778708
2404  1367003837  Stress_u41     1  43.70543645,-72.28828558
2405  1367193608  Stress_u41     2  43.70535262,-72.28819131
2406  1367312977  Stress_u41     3  43.70526777,-72.28881679
2407  1368000327  Stress_u41     3                   Unknown

[2408 rows x 4 columns]

After converting resp_time to timestamp:
       resp_time  student_id level                  location  \
0     1364